# Download Feature Layer Data from Hugging Face

Download only the `02_feature_layer/training/outputs` files from the PropertyLens/Resealeflats dataset using the HF token from .env file.

In [1]:
!pip install python-dotenv huggingface-hub

## Setup and Configuration

In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv
from huggingface_hub import hf_hub_download, HfApi

# ── Resolve repo root (works from repo root OR any sublayer folder) ──────────
cwd = Path.cwd().resolve()
REPO_ROOT = cwd if (cwd / "hf_data").exists() or (cwd / "01_data_layer").exists() else cwd.parent

# Load environment variables
load_dotenv(REPO_ROOT / ".env", override=False)
load_dotenv(cwd / ".env", override=False)
hf_token = os.getenv("HF_TOKEN")

if not hf_token:
    raise ValueError("HF_TOKEN not found in .env file")

# Dataset info
repo_id   = "PropertyLens/Resealeflats"
repo_type = "dataset"

# Output directory (mirrors HF repo structure)
output_dir = REPO_ROOT / "hf_data"
output_dir.mkdir(exist_ok=True)

print(f"Repo root  : {REPO_ROOT}")
print(f"Output dir : {output_dir.absolute()}")
print(f"HF repo    : {repo_id}\n")

## Fetch and Filter Feature Layer Files

In [ ]:
try:
    api = HfApi()

    print("Fetching file list from repository...")
    repo_info = api.repo_info(repo_id=repo_id, repo_type=repo_type, token=hf_token)
    all_repo_files = [f.rfilename for f in repo_info.siblings if f.rfilename]

    # 02_feature_layer outputs
    feature_files = sorted(
        f for f in all_repo_files
        if f.startswith('02_feature_layer/training/outputs/')
    )

    # 05_photo_layer artifacts (model weights, metadata, training curves)
    artifact_files = sorted(
        f for f in all_repo_files
        if f.startswith('05_photo_layer/artifacts/')
    )

    print(f"02_feature_layer files  : {len(feature_files)}")
    for fp in feature_files:
        print(f"  • {fp}")

    print(f"\n05_photo_layer artifacts: {len(artifact_files)}")
    for fp in artifact_files:
        print(f"  • {fp}")

except Exception as e:
    print(f"Error fetching file list: {str(e)}")
    raise

## Download Files

In [ ]:
def _download_files(files: list[str], label: str) -> tuple[int, int]:
    """Download a list of HF repo paths into output_dir. Returns (ok, failed)."""
    ok, failed = 0, 0
    print(f"{'=' * 70}")
    print(f"Downloading {label} ({len(files)} files)...\n")
    for idx, file_path in enumerate(files, 1):
        print(f"[{idx}/{len(files)}] {file_path}")
        try:
            local_path = hf_hub_download(
                repo_id=repo_id,
                filename=file_path,
                repo_type=repo_type,
                local_dir=output_dir,
                token=hf_token,
                force_download=False,
            )
            size_mb = Path(local_path).stat().st_size / (1024 * 1024)
            print(f"         ✓ ({size_mb:.2f} MB)\n")
            ok += 1
        except Exception as e:
            print(f"         ✗ Error: {str(e)}\n")
            failed += 1
    return ok, failed


feat_ok,    feat_failed    = _download_files(feature_files,  "02_feature_layer/training/outputs")
artifact_ok, artifact_failed = _download_files(artifact_files, "05_photo_layer/artifacts")

## Download Summary

In [ ]:
print("=" * 70)
print("Download Summary")
print("=" * 70)
print(f"  02_feature_layer/training/outputs : ✓ {feat_ok}  ✗ {feat_failed}")
print(f"  05_photo_layer/artifacts          : ✓ {artifact_ok}  ✗ {artifact_failed}")
total_ok  = feat_ok  + artifact_ok
total_err = feat_failed + artifact_failed
print(f"\n  Total: ✓ {total_ok} downloaded   ✗ {total_err} failed")
print(f"\nFiles saved under: {output_dir.absolute()}")

## Downloaded Directory Structure

In [ ]:
def _list_dir(d: Path, label: str) -> None:
    print(f"\n  {label}")
    print(f"  {'─' * 60}")
    if d.exists():
        files = sorted(f for f in d.glob('*') if f.is_file())
        for f in files:
            size_mb = f.stat().st_size / (1024 * 1024)
            print(f"    {f.name:<50s}  {size_mb:>7.2f} MB")
        if not files:
            print("    (empty)")
    else:
        print("    (directory not found)")

print("Downloaded directory structure:")
print("=" * 70)
_list_dir(output_dir / "02_feature_layer" / "training" / "outputs", "02_feature_layer/training/outputs")
_list_dir(output_dir / "05_photo_layer"   / "artifacts",            "05_photo_layer/artifacts")

# Download Feature Layer Data from Hugging Face

Download only the 02_feature_layer/training/outputs files from the PropertyLens/Resealeflats dataset using HF token from .env file.